# Métricas de evaluación: BLEU y ROUGE

Cálculo paso a paso de BLEU (`nltk`) y ROUGE (`rouge_score` y `evaluate`) sobre frases cortas en español e inglés. La teoría está en el [README](README.md).

## BLEU score

In [1]:
from nltk.translate import bleu_score

In [2]:
def read_lines(f):
    line1 = f.readline().strip("\n")
    if not line1:
        return None
    line2 = f.readline().strip("\n")
    line3 = f.readline().strip("\n")
    return [line1, line2, line3]

In [3]:
def split_and_pack(lines: list[str]):
    reference1 = lines[0].split()
    reference2 = lines[1].split()
    hypothesis = lines[2].split()
    return [[reference1, reference2], hypothesis]


In [4]:
def bleu_step_by_step(references, hypothesis, weights, smoothing_function = None) -> float:

    hyp_len = len(hypothesis)
    closest_ref_length = bleu_score.closest_ref_length(references, hyp_len)
    print(f"Closest reference length: {closest_ref_length}")
    brevity_penalty = bleu_score.brevity_penalty(closest_ref_length, hyp_len)
    print(f"Brevity penalty: {brevity_penalty}")
    corpus_bleu = bleu_score.corpus_bleu([references], [hypothesis], weights)
    print(f"Corpus bleu: {corpus_bleu}")
    sentence_bleu = bleu_score.sentence_bleu(references, hypothesis, weights)
    print(f"Sentence bleu: {sentence_bleu}")
    return sentence_bleu

In [5]:
#Prueba inicial con BLEU
reference1 = "El perro come pienso de pavo".split()
reference2 = "El perro consume pienso de pavo".split()
hypothesis = "El perro cena pienso de pavo".split()


references = [reference1, reference2]
weights = (0.5, 0.5, 0, 0)

bleu_step_by_step(references, hypothesis, weights)

Closest reference length: 6
Brevity penalty: 1.0
Corpus bleu: 0.7071067811865476
Sentence bleu: 0.7071067811865476


0.7071067811865476

In [6]:
#Como las oraciones son sencillas no hay 3-gramas ni 4-gramas. Sus pesos se inicializan a 0 para que el score tenga un valor significativo (seria ~0 en otro caso)
weights = (0.5, 0.5, 0, 0)

with open("mini_dataset.txt", 'r') as f:
    while True:
        print("-------------------------------------------")
        lines = read_lines(f)
        if not lines:
            break
        print(f"References: {lines[0]}, {lines[1]}")
        print(f"Hypothesis: {lines[2]}")
        references, hypothesis = split_and_pack(lines)
        bleu_step_by_step(references, hypothesis, weights)

-------------------------------------------
References: El perro come pienso de pavo, El perro consume pienso de pavo
Hypothesis: El perro cena pienso de pavo
Closest reference length: 6
Brevity penalty: 1.0
Corpus bleu: 0.7071067811865476
Sentence bleu: 0.7071067811865476
-------------------------------------------
References: El perro come pienso de pavo, El perro consume pienso de pavo
Hypothesis: El perro perro perro perro
Closest reference length: 6
Brevity penalty: 0.8187307530779819
Corpus bleu: 0.25890539701513365
Sentence bleu: 0.25890539701513365
-------------------------------------------
References: El hombre de sombrero azul bajo la colina rapidamente, El hombre que vestia un sombrero azul descendio la colina rapidamente
Hypothesis: El hombre con sombrero bajo la colina
Closest reference length: 9
Brevity penalty: 0.7514772930752859
Corpus bleu: 0.4919573683654302
Sentence bleu: 0.4919573683654302
-------------------------------------------
References: El hombre de sombrer

## ROUGE

### rouge_score

In [7]:
from rouge_score import rouge_scorer

In [8]:
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

In [9]:
def print_rouge_results(results):
    for metric, score in results.items():
        print(f"{metric}:")
        print(f"    Precision: {score.precision}")
        print(f"    Recall: {score.recall}")
        print(f"    F1: {score.fmeasure}")

In [10]:
def rouge_step_by_step(reference, hypothesis):
    print(f"Reference: {reference}")
    print(f"Hypothesis: {hypothesis}")
    scorer_no_stemmer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
    results_no_stemmer = scorer_no_stemmer.score(reference, hypothesis)
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    results = scorer.score(reference, hypothesis)
    print("---No Stemmer---")
    print_rouge_results(results_no_stemmer)
    print("---With Stemmer---")
    print_rouge_results(results)


In [11]:
reference1 = "El perro come pienso de pavo"
hypothesis = "El perro cena pienso de pavo"
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
print_rouge_results(scorer.score(reference1, hypothesis))


rouge1:
    Precision: 0.8333333333333334
    Recall: 0.8333333333333334
    F1: 0.8333333333333334
rouge2:
    Precision: 0.6
    Recall: 0.6
    F1: 0.6
rougeL:
    Precision: 0.8333333333333334
    Recall: 0.8333333333333334
    F1: 0.8333333333333334


#### Uso del stemmer

In [12]:
reference1 = "Playing cooking dancing participating walking"
reference2 = "Played cooked danced participated walked"
hypothesis = "Play cook dance participate walk"
rouge_step_by_step(reference1, hypothesis)
rouge_step_by_step(reference2, hypothesis)

Reference: Playing cooking dancing participating walking
Hypothesis: Play cook dance participate walk
---No Stemmer---
rouge1:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rouge2:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rougeL:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
---With Stemmer---
rouge1:
    Precision: 1.0
    Recall: 1.0
    F1: 1.0
rouge2:
    Precision: 1.0
    Recall: 1.0
    F1: 1.0
rougeL:
    Precision: 1.0
    Recall: 1.0
    F1: 1.0
Reference: Played cooked danced participated walked
Hypothesis: Play cook dance participate walk
---No Stemmer---
rouge1:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rouge2:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rougeL:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
---With Stemmer---
rouge1:
    Precision: 1.0
    Recall: 1.0
    F1: 1.0
rouge2:
    Precision: 1.0
    Recall: 1.0
    F1: 1.0
rougeL:
    Precision: 1.0
    Recall: 1.0
    F1: 1.0


El *stemmer* de `rouge_score` es el de Porter, pensado para **inglés**: reduce `playing`/`played`/`play` a la misma raíz, por eso en el ejemplo anterior la puntuación pasa de 0 a 1 al activarlo. En español no reconoce las terminaciones (`jugando`, `jugamos`, `jugar`), así que activarlo o no da el mismo resultado, como se ve a continuación.

In [13]:
reference1 = "Jugando cocinando bailando participando corriendo"
reference2 = "Jugamos cocinamos bailamos participamos corrimos"
hypothesis = "Jugar cocinar bailar particir correr"
rouge_step_by_step(reference1, hypothesis)
rouge_step_by_step(reference2, hypothesis)

Reference: Jugando cocinando bailando participando corriendo
Hypothesis: Jugar cocinar bailar particir correr
---No Stemmer---
rouge1:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rouge2:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rougeL:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
---With Stemmer---
rouge1:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rouge2:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rougeL:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
Reference: Jugamos cocinamos bailamos participamos corrimos
Hypothesis: Jugar cocinar bailar particir correr
---No Stemmer---
rouge1:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rouge2:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rougeL:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
---With Stemmer---
rouge1:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rouge2:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0
rougeL:
    Precision: 0.0
    Recall: 0.0
    F1: 0.0


### evaluate

In [14]:
import evaluate

In [15]:
rouge = evaluate.load('rouge')

In [16]:
# evaluate espera LISTAS de predicciones y referencias (una por ejemplo).
# Si se le pasan strings sueltos, los recorre carácter a carácter y el resultado no tiene sentido.
reference1 = "El perro come pienso de pavo"
hypothesis = "El perro cena pienso de pavo"

results = rouge.compute(predictions = [hypothesis], references = [reference1])
results

{'rouge1': np.float64(0.8333333333333334),
 'rouge2': np.float64(0.6),
 'rougeL': np.float64(0.8333333333333334),
 'rougeLsum': np.float64(0.8333333333333334)}